# Chapter 3 (부록) — 멀티 헤드 어텐션 시각화 : 코드 해설 (Python 초보용)

> `Chapter_3_Excercise_Viz_Multi_head_attention.ipynb` 의 원본 셀은 그대로 두고, 번호가 붙은 셀마다 바로 위에 설명을 넣은 해설판입니다.

## 이 노트북에서 하는 일 (한 문장)

앞 노트북에서 직접 만든 어텐션이 **실제로 학습된 GPT-2 안에서는 어디를 보고 있는지** 그림으로 확인합니다.

```
 sentence -> tokens -> GPT-2 (12 layers x 12 heads)
                           |
                           |  hook : peek at attention weights of every layer
                           v
                    all_attentions  ->  heatmap (cell 2) / line view (cell 3)
```

| 셀 | 내용 |
|---|---|
| 1 | GPT-2 를 불러와 문장을 넣고, 모든 층의 어텐션 가중치를 수집 |
| 2 | 한 층의 모든 헤드를 **히트맵**(색칠한 표)으로 그리기 |
| 3 | 층/헤드를 고르는 **드롭다운**과 함께 선(line)으로 그리기 |

### 실행 전 준비물
- `models/gpt2/gpt2-small-124M.pth` (사전 학습된 GPT-2 가중치 파일. 없으면 자동으로 내려받습니다)
- `matplotlib`, `ipywidgets` 패키지. 저장된 셀 1의 출력에 `ModuleNotFoundError: No module named 'matplotlib'` 가 남아 있는데, 이는 마지막 실행 때 그 환경에 matplotlib 이 설치되어 있지 않았다는 뜻입니다. 같은 에러가 나면 터미널에서 `uv add matplotlib ipywidgets` (또는 `pip install matplotlib ipywidgets`) 로 설치한 뒤 다시 실행하세요.

# Multi-Head Attention Visualized

---
## 셀 1 — GPT-2 불러오기 + 어텐션 가중치 수집

### 🎯 이 셀이 하는 일
1. 사전 학습된 GPT-2 Small 모델을 불러오고
2. 예문을 토큰으로 바꿔 모델에 넣은 다음
3. **훅(hook)** 이라는 장치로 각 층의 어텐션 가중치를 몰래 꺼내 `all_attentions` 리스트에 모읍니다.

### 코드 뜯어보기 — ① 준비

```python
import sys
sys.path.append('.')
from previous_chapters import load_gpt2_model
```
- `sys.path` 는 파이썬이 모듈을 찾아보는 폴더 목록입니다. `'.'`(현재 폴더)를 추가해서, 같은 폴더의 `previous_chapters.py` 를 `import` 할 수 있게 합니다.
- `previous_chapters.py` 는 앞 챕터들에서 만든 클래스·함수(`GPTModel`, `MultiHeadAttention` 등)를 모아 둔 파일입니다.

```python
GPT_CONFIG_SMALL = {
    "vocab_size": 50257, "context_length": 1024, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.0, "qkv_bias": True
}
```
`{ "이름": 값, ... }` 은 **딕셔너리(dict)** 입니다. 이름표(키)로 값을 꺼내는 자료형이며, 모델의 설정값을 한데 모아 둔 것입니다.

| 키 | 뜻 |
|---|---|
| `vocab_size` | 단어장 크기 (토큰 종류 수) |
| `context_length` | 한 번에 볼 수 있는 최대 토큰 수 |
| `emb_dim` | 토큰 하나를 표현하는 벡터 길이 |
| `n_heads` | 어텐션 헤드 수 |
| `n_layers` | 트랜스포머 블록(층)을 몇 개 쌓는지 |
| `drop_rate` | 드롭아웃 비율 (0.0 = 끔) |
| `qkv_bias` | Q/K/V 레이어에 편향을 쓸지 (공개된 GPT-2 가중치는 True) |

```python
model = load_gpt2_model("gpt2-small-124M.pth", GPT_CONFIG_SMALL)
model.eval()
```
- `load_gpt2_model` : 설정대로 빈 모델을 만들고, 파일에 저장된 **학습된 숫자(가중치)** 를 채워 넣습니다.
- `model.eval()` : **평가 모드**로 바꿉니다. 드롭아웃처럼 학습 때만 쓰는 무작위 동작을 꺼서 결과가 매번 같게 합니다.

```python
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(sentence_a)
tokens = [tokenizer.decode([t]) for t in token_ids]
input_tensor = torch.tensor([token_ids])
```
- `token_ids` : 문장을 토큰 ID 리스트로
- `tokens` : ID 를 하나씩 다시 글자로 바꾼 리스트 (그림의 축 이름표로 사용)
- `torch.tensor([token_ids])` : 대괄호로 한 번 더 감싸 `[1, 토큰 수]` 모양으로 만듭니다. 모델은 항상 **배치** 형태(문장 여러 개)를 입력으로 받기 때문에, 문장이 1개여도 "1개짜리 배치"로 만들어 줘야 합니다.

### 코드 뜯어보기 — ② 훅(hook)

**훅이란?** 모델 안의 특정 부품에 **"네가 계산을 마칠 때마다 이 함수도 불러 줘"** 라고 걸어 두는 갈고리입니다. 모델 코드를 고치지 않고도 중간 과정을 들여다볼 수 있습니다.

우리 `MultiHeadAttention` 은 최종 결과(문맥 벡터)만 돌려주고 **어텐션 가중치는 밖으로 내보내지 않습니다.** 그래서 훅 안에서 같은 입력으로 가중치를 **한 번 더 계산**해서 저장합니다.

```python
all_attentions = []

def make_hook():
    def hook_fn(module, input, output):
        x = input[0]
        b, num_tokens, _ = x.shape
        keys = module.W_key(x)
        queries = module.W_query(x)
        keys = keys.view(b, num_tokens, module.num_heads, module.head_dim).transpose(1, 2)
        queries = queries.view(...).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = module.mask.bool()[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        all_attentions.append(attn_weights.detach())
    return hook_fn
```
- **함수 안의 함수** : `make_hook()` 을 부르면 안쪽의 `hook_fn` 함수 자체를 돌려줍니다. "훅 함수를 찍어내는 공장"입니다.
- `hook_fn(module, input, output)` : PyTorch 가 정한 훅의 모양입니다. `module` = 훅이 걸린 부품(어텐션), `input` = 그 부품에 들어간 입력, `output` = 나온 결과.
- `x = input[0]` : `input` 은 튜플로 전달되므로 첫 번째 것을 꺼냅니다.
- `b, num_tokens, _ = x.shape` : 세 번째 값은 쓰지 않아서 `_` 로 받았습니다.
- 가운데 계산은 앞 노트북 **셀 4의 `forward` 와 똑같습니다** (Q·K 만들기 → 헤드로 쪼개기 → 점수 → 마스크 → softmax). Value 는 가중치를 구하는 데 필요 없어서 뺐습니다.
- `.detach()` : "학습용 계산 기록을 떼어 내고 숫자만" 저장합니다.
- 저장되는 텐서 모양 : `[1, 헤드 수, 토큰 수, 토큰 수]`

### 코드 뜯어보기 — ③ 훅 걸고, 실행하고, 떼기

```python
hooks = [block.att.register_forward_hook(make_hook()) for block in model.trf_blocks]
with torch.no_grad():
    model(input_tensor)
for hook in hooks:
    hook.remove()
```
- `model.trf_blocks` : 12개의 트랜스포머 블록. 각 블록의 `.att` 가 어텐션 부품입니다.
- `register_forward_hook(함수)` : 훅을 겁니다. 12개 블록 모두에 걸고, 나중에 떼기 위한 손잡이를 `hooks` 리스트에 모아 둡니다.
- `with torch.no_grad():` : "이 안에서는 학습용 기록을 하지 마라". 구경만 할 때 메모리와 시간을 아낍니다.
- `model(input_tensor)` : 모델을 한 번 실행합니다. 결과는 버리지만, 실행되는 동안 **12개의 훅이 차례로 불려** `all_attentions` 에 12개 텐서가 쌓입니다.
- `hook.remove()` : 다 썼으면 훅을 뗍니다. 안 떼면 모델을 실행할 때마다 계속 쌓입니다.

### 실행 결과 읽기
```python
print(f"레이어 수: {len(all_attentions)}, 헤드 수: {all_attentions[0].shape[1]}, 토큰: {tokens}")
```
정상 실행되면 `레이어 수: 12, 헤드 수: 12` 와 함께 문장이 쪼개진 토큰 목록이 출력됩니다. 토큰 앞의 공백(`' artist'`)은 GPT-2 토크나이저가 띄어쓰기를 **뒤 단어의 앞에 붙여서** 한 토큰으로 다루기 때문입니다.

In [1]:
# 1
import sys
sys.path.append('.')
import torch
import tiktoken
import matplotlib.pyplot as plt
from previous_chapters import load_gpt2_model

# GPT-2 Small (124M) 설정
GPT_CONFIG_SMALL = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True
}

sentence_a = "The artist painted the portrait of a woman with a brush"

# 모델 로드
model = load_gpt2_model("gpt2-small-124M.pth", GPT_CONFIG_SMALL)
model.eval()

# 토크나이저
tokenizer = tiktoken.get_encoding("gpt2")

# 토큰화 및 attention weights 수집 (forward hook)
token_ids = tokenizer.encode(sentence_a)
tokens = [tokenizer.decode([t]) for t in token_ids]
input_tensor = torch.tensor([token_ids])

all_attentions = []

def make_hook():
    def hook_fn(module, input, output):
        x = input[0]
        b, num_tokens, _ = x.shape
        keys = module.W_key(x)
        queries = module.W_query(x)
        keys = keys.view(b, num_tokens, module.num_heads, module.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, module.num_heads, module.head_dim).transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = module.mask.bool()[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        all_attentions.append(attn_weights.detach())
    return hook_fn

hooks = [block.att.register_forward_hook(make_hook()) for block in model.trf_blocks]
with torch.no_grad():
    model(input_tensor)
for hook in hooks:
    hook.remove()

print(f"레이어 수: {len(all_attentions)}, 헤드 수: {all_attentions[0].shape[1]}, 토큰: {tokens}")

ModuleNotFoundError: No module named 'matplotlib'

---
## 셀 2 — 한 층의 모든 헤드를 히트맵으로 그리기

### 🎯 이 셀이 하는 일
4번 층(`layer_idx = 4`)의 헤드들이 각각 어디에 주목하는지, **헤드마다 색칠한 표(히트맵) 한 장씩** 그립니다.

### 코드 뜯어보기

```python
layer_idx = 4
num_heads = all_attentions[layer_idx].shape[1]
```
`all_attentions[4]` 는 `[1, 헤드 수, T, T]` 모양이므로 `shape[1]` 이 헤드 수입니다.

> 📝 주석의 `# 16` 과 아래 제목의 `"GPT-2 Medium"` 은 예전에 Medium 모델(16 헤드)로 실습하던 흔적입니다. 셀 1에서 불러온 것은 **GPT-2 Small 이므로 실제 값은 12** 입니다. 코드는 `num_heads` 를 직접 읽어 오므로 문제없이 동작하고, 그림 제목의 글자만 실제와 다릅니다.

```python
fig, axes = plt.subplots(2, num_heads // 2, figsize=(24, 8))
axes = axes.flatten()
```
- `plt.subplots(행, 열)` : 여러 칸짜리 그림판을 만듭니다. 2행 × 6열 = 12칸. `fig` 는 종이 전체, `axes` 는 각 칸들입니다.
- `num_heads // 2` : `//` 는 몫만 구하는 나눗셈(12 // 2 = 6).
- `figsize=(24, 8)` : 가로 24, 세로 8 인치.
- `axes.flatten()` : 2×6 표 모양으로 온 칸들을 **한 줄(12개)** 로 펴서 `axes[0]` ~ `axes[11]` 로 쉽게 쓰게 합니다.

```python
for head_idx in range(num_heads):
    attn = all_attentions[layer_idx][0, head_idx].numpy()
    axes[head_idx].imshow(attn, cmap="Blues")
    axes[head_idx].set_title(f"Head {head_idx}", fontsize=9)
    axes[head_idx].set_xticks(range(len(tokens)))
    axes[head_idx].set_xticklabels(tokens, rotation=90, fontsize=7)
    axes[head_idx].set_yticks(range(len(tokens)))
    axes[head_idx].set_yticklabels(tokens, fontsize=7)
```
- `[0, head_idx]` : 배치의 0번째(문장이 하나뿐), 그중 `head_idx` 번 헤드 → `[T, T]` 표 한 장
- `.numpy()` : 텐서를 넘파이 배열로 바꿉니다. 그림 도구(matplotlib)는 넘파이를 씁니다.
- `imshow(attn, cmap="Blues")` : 숫자 표를 그림으로. 값이 **클수록 진한 파랑**.
- `set_xticks` / `set_xticklabels` : 가로축 눈금 위치와 이름표(토큰 글자). `rotation=90` 은 글자를 세로로 세움.

```python
plt.suptitle(...)     # 그림 전체 제목
plt.tight_layout()    # 칸들이 겹치지 않게 자동 정리
plt.show()            # 화면에 표시
```

### 그림 읽는 법
```
              looked-at token (column)
              The  artist  painted  the ...
 The         [ ## ]
 artist      [ #    ## ]
 painted     [ .    ##     # ]
 the         [ .    .      #     # ]
 (row = token that is looking)
```
- **행** = 지금 보고 있는 토큰, **열** = 바라보는 대상 토큰
- 색이 진한 칸 = 많이 주목하는 곳
- **오른쪽 위 삼각형은 항상 비어 있습니다** → 미래를 가리는 causal mask 가 실제로 작동하고 있다는 증거입니다.
- 헤드마다 무늬가 다릅니다. 어떤 헤드는 바로 앞 토큰만(대각선 바로 아래), 어떤 헤드는 첫 토큰만(맨 왼쪽 열) 봅니다. "헤드마다 관점이 다르다"를 눈으로 확인하는 것이 이 셀의 목적입니다.

In [ ]:
# 2
# head_view 대체: layer=4의 모든 헤드 attention 히트맵
layer_idx = 4
num_heads = all_attentions[layer_idx].shape[1]  # 16

fig, axes = plt.subplots(2, num_heads // 2, figsize=(24, 8))
axes = axes.flatten()

for head_idx in range(num_heads):
    attn = all_attentions[layer_idx][0, head_idx].numpy()
    axes[head_idx].imshow(attn, cmap="Blues")
    axes[head_idx].set_title(f"Head {head_idx}", fontsize=9)
    axes[head_idx].set_xticks(range(len(tokens)))
    axes[head_idx].set_xticklabels(tokens, rotation=90, fontsize=7)
    axes[head_idx].set_yticks(range(len(tokens)))
    axes[head_idx].set_yticklabels(tokens, fontsize=7)

plt.suptitle(f"Head View - Layer {layer_idx} (GPT-2 Medium)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 셀 3 — 드롭다운으로 층/헤드를 골라 선으로 보기

### 🎯 이 셀이 하는 일
층(Layer)과 헤드(Head)를 **드롭다운 메뉴로 고르면** 그 헤드의 어텐션을 "왼쪽 토큰 → 오른쪽 토큰"을 잇는 **선**으로 다시 그려 주는 작은 대화형 화면을 만듭니다. 선이 진할수록 강하게 주목한다는 뜻입니다.

### 코드 뜯어보기 — ① 준비와 이전 화면 정리

```python
import io
import numpy as np
import ipywidgets as widgets
from IPython.display import display
```
`ipywidgets` 는 노트북 안에 버튼·드롭다운 같은 조작 도구를 넣어 주는 라이브러리입니다.

```python
previous_ui = globals().get("_attention_ui")
if previous_ui is not None:
    previous_ui["controls"].close()
    ...
```
- `globals()` : 지금 노트북에 정의된 모든 전역 변수를 담은 딕셔너리입니다. `.get("이름")` 은 그 이름의 변수가 있으면 꺼내고 없으면 `None` 을 줍니다.
- 이 셀을 **두 번째 실행**할 때 예전에 만든 위젯이 남아 화면이 겹치는 것을 막으려고, 이전 것이 있으면 닫습니다. (셀 맨 끝에서 `globals()["_attention_ui"] = {...}` 로 이번에 만든 위젯들을 기억시켜 둡니다.)

```python
plot_widget = widgets.Image(format="png")
```
그림을 보여 줄 빈 액자를 하나 만듭니다.

### 코드 뜯어보기 — ② 그리는 함수 `plot_attention_lines`

```python
def plot_attention_lines(change=None):
    layer_idx = layer_w.value
    head_idx = head_w.value
    attn = all_attentions[layer_idx][0, head_idx].numpy()
    n = len(tokens)
```
- `change=None` : 드롭다운 값이 바뀔 때 위젯이 "무엇이 바뀌었는지" 정보를 넘겨주는데, 여기서는 쓰지 않습니다. 기본값을 `None` 으로 두어 인자 없이도 부를 수 있게 했습니다.
- `layer_w.value`, `head_w.value` : 드롭다운에서 **현재 선택된 값**. (`layer_w` 는 함수 아래쪽에서 만들어지지만, 함수가 **실제로 불리는 시점**에는 이미 만들어져 있으므로 문제없습니다.)

```python
    with plt.ioff():
        fig, ax = plt.subplots(figsize=(8, max(6, n * 0.45)))
    fig.patch.set_facecolor("#000000")
    ax.set_facecolor("#000000")
```
- `plt.ioff()` : 그림이 자동으로 화면에 튀어나오지 않게 잠시 끕니다(우리는 액자에 직접 넣을 것이므로).
- `max(6, n * 0.45)` : 토큰이 많을수록 세로로 길게, 최소 6.
- `"#000000"` : 검정색(배경).

```python
    y_pos = np.linspace(0.95, 0.05, n)
    x_left, x_right = 0.2, 0.8
```
`np.linspace(시작, 끝, 개수)` 는 시작~끝을 **같은 간격으로 나눈 숫자 n개**를 만듭니다. 토큰들을 위(0.95)에서 아래(0.05)로 고르게 배치할 세로 위치입니다.

```python
    for i in range(n):
        for j in range(n):
            w = float(attn[i, j])
            if w > 0.005:
                ax.plot([x_left, x_right], [y_pos[i], y_pos[j]],
                        color="#4A90D9", alpha=min(w * 2, 1.0), linewidth=1.2)
```
- **이중 for 문** : 모든 (i, j) 조합을 돕니다. i = 보는 토큰(왼쪽), j = 보이는 토큰(오른쪽).
- `if w > 0.005` : 가중치가 거의 0 인 선은 그리지 않습니다(미래 쪽은 0 이므로 자동으로 빠짐).
- `ax.plot([x1, x2], [y1, y2])` : 두 점을 잇는 선.
- `alpha=min(w * 2, 1.0)` : 투명도. 가중치가 클수록 진하게, 단 1.0 을 넘지 않게 `min` 으로 제한.

```python
    for i, tok in enumerate(tokens):
        ax.text(x_left - 0.01, y_pos[i], tok, ha="right", ...)
        ax.text(x_right + 0.01, y_pos[i], tok, ha="left", ...)
```
`enumerate(리스트)` 는 **(번호, 값)** 을 함께 꺼내 줍니다. 왼쪽과 오른쪽에 토큰 글자를 적습니다. `ha` 는 가로 정렬(horizontal alignment)입니다.

```python
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", ...)
    plot_widget.value = buffer.getvalue()
    buffer.close()
    plt.close(fig)
```
그림을 파일로 저장하는 대신 **메모리 안의 임시 공간**(`BytesIO`)에 PNG 로 저장하고, 그 내용을 액자(`plot_widget`)에 넣습니다. 다 쓴 그림은 `plt.close` 로 닫아 메모리를 돌려줍니다.

### 코드 뜯어보기 — ③ 드롭다운 만들고 연결하기

```python
layer_w = widgets.Dropdown(options=list(range(len(all_attentions))), value=4, description="Layer:")
head_w  = widgets.Dropdown(options=list(range(all_attentions[0].shape[1])), value=3, description="Head:")
controls = widgets.HBox([layer_w, head_w])
```
- `options` : 고를 수 있는 값 목록 (층 0~11, 헤드 0~11), `value` : 처음 선택값
- `HBox` : 위젯들을 **가로로 나란히** 놓는 상자

```python
layer_w.observe(plot_attention_lines, names="value")
head_w.observe(plot_attention_lines, names="value")
```
`observe(함수, names="value")` : "**값이 바뀌면 이 함수를 불러라**"라고 등록합니다. 함수 이름 뒤에 괄호가 없다는 점에 주의하세요. 지금 실행하는 것이 아니라 **함수 자체를 맡겨 두는 것**입니다(이런 함수를 콜백이라고 합니다).

```python
display(controls)
display(plot_widget)
...
plot_attention_lines()
```
드롭다운과 액자를 화면에 띄우고, 맨 처음 한 번은 직접 불러서 첫 그림을 그립니다. (마지막 줄의 주석 `# ?? ???` 는 한글이 깨진 흔적으로, 원래 "초기 표시" 정도의 메모였을 것입니다. 실행에는 영향이 없습니다.)

### 그림 읽는 법
- 왼쪽 단어에서 오른쪽 단어로 선이 그어져 있으면 "왼쪽 단어가 오른쪽 단어를 참고한다"는 뜻입니다.
- 선은 항상 **같은 높이이거나 위쪽**(= 자기 자신이나 앞 단어)으로만 향합니다. 미래는 못 보기 때문입니다.
- 층과 헤드를 바꿔 보면서, 예를 들어 `brush` 가 `painted` 나 `with` 를 진하게 보는 헤드가 있는지 찾아보세요.

In [4]:
# 3
import io
import numpy as np
import ipywidgets as widgets
from IPython.display import display

previous_ui = globals().get("_attention_ui")
if previous_ui is not None:
    previous_ui["controls"].close()
    previous_ui["layer_w"].close()
    previous_ui["head_w"].close()
    previous_ui["plot_widget"].close()

plot_widget = widgets.Image(format="png")

def plot_attention_lines(change=None):
    layer_idx = layer_w.value
    head_idx = head_w.value
    attn = all_attentions[layer_idx][0, head_idx].numpy()
    n = len(tokens)

    with plt.ioff():
        fig, ax = plt.subplots(figsize=(8, max(6, n * 0.45)))

    fig.patch.set_facecolor("#000000")
    ax.set_facecolor("#000000")

    y_pos = np.linspace(0.95, 0.05, n)
    x_left, x_right = 0.2, 0.8

    for i in range(n):
        for j in range(n):
            w = float(attn[i, j])
            if w > 0.005:
                ax.plot([x_left, x_right], [y_pos[i], y_pos[j]],
                        color="#4A90D9", alpha=min(w * 2, 1.0), linewidth=1.2)

    for i, tok in enumerate(tokens):
        ax.text(x_left - 0.01, y_pos[i], tok, ha="right", va="center",
                color="white", fontsize=10, fontfamily="monospace")
        ax.text(x_right + 0.01, y_pos[i], tok, ha="left", va="center",
                color="white", fontsize=10, fontfamily="monospace")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(f"Layer: {layer_idx}  Head: {head_idx}",
                 color="white", fontsize=12, pad=10)
    fig.tight_layout()

    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", facecolor=fig.get_facecolor(), bbox_inches="tight")
    plot_widget.value = buffer.getvalue()
    buffer.close()
    plt.close(fig)

layer_w = widgets.Dropdown(
    options=list(range(len(all_attentions))), value=4, description="Layer:")
head_w = widgets.Dropdown(
    options=list(range(all_attentions[0].shape[1])), value=3, description="Head:")
controls = widgets.HBox([layer_w, head_w])

layer_w.observe(plot_attention_lines, names="value")
head_w.observe(plot_attention_lines, names="value")

display(controls)
display(plot_widget)
globals()["_attention_ui"] = {
    "layer_w": layer_w,
    "head_w": head_w,
    "controls": controls,
    "plot_widget": plot_widget,
}
plot_attention_lines()  # ?? ???


Image(value=b'')

---
## 📌 정리

1. **훅(hook)** 을 쓰면 모델 코드를 고치지 않고도 중간 계산 결과를 꺼내 볼 수 있다.
2. 어텐션 가중치는 `[층][배치, 헤드, 보는 토큰, 보이는 토큰]` 구조다.
3. 히트맵의 오른쪽 위가 비어 있는 것 = **causal mask** 가 작동 중.
4. 헤드마다 주목하는 패턴이 다르다 = **멀티 헤드**를 쓰는 이유.